In [23]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.neighbors import NearestNeighbors

In [2]:
hw = pd.read_csv('homework_2.1.csv')
hw

,time,G1,G2,G3
0,0,0.882026,1.441575,0.065409
1,1,0.210079,-0.163880,0.140310
2,2,0.509369,-0.115242,0.819830
3,3,1.150447,1.014698,0.607632
4,4,0.973779,-0.046562,0.610066
...,...,...,...,...
95,95,1.303287,1.364227,1.768446
96,96,0.965250,1.845895,1.258862
97,97,1.862935,1.881752,1.511477
98,98,1.043456,2.561618,1.030275


# Question 1


In [13]:
# 1. Reshape the dataframe to long format
# This creates a 'group' column (G1, G2, G3) and an 'outcome' column
df = hw.melt(id_vars=['time'], 
                  value_vars=['G1', 'G2', 'G3'], 
                  var_name='group', 
                  value_name='outcome')

# 2. Create dummy variables for the groups
# drop_first=False allows you to see the coefficient for every group
group_dummies = pd.get_dummies(df['group'], drop_first=False).astype(int)

# 3. Define predictors (Treatment 'time' + Group fixed effects)
X = pd.concat([df['time'], group_dummies], axis=1)
# X = df.drop('outcome',axis=1).groupby('group')
print("==============X-dummy=====================")
print(X)

# 4. Run the OLS Regression
# We exclude a global constant to get the absolute fixed effect for each group
model = sm.OLS(df['outcome'], X).fit()


# View the coefficients for G1, G2, and G3
print("===============================Model==========================================")
print(model.summary())
for col in hw.columns:
    if col != 'time':
        df = hw.copy()
        df['time_m'] = hw['time'] - hw['time'].mean()
        df[f'{col}_m'] = hw[col] - hw[col].mean()
        model = sm.OLS(hw[col], hw['time']).fit()
        model_mean = sm.OLS(df[f'{col}_m'], df['time_m']).fit()
        print("==============================================================================")
        print(f'Model for {col} Raw')
        # print(hw[col])
        print(model.summary())
        print('***************************shifted********************************************')
        print("==============================================================================")
        print(model_mean.summary())
df


==============X-dummy=====================
     time  G1  G2  G3
0       0   1   0   0
1       1   1   0   0
2       2   1   0   0
3       3   1   0   0
4       4   1   0   0
..    ...  ..  ..  ..
295    95   0   0   1
296    96   0   0   1
297    97   0   0   1
298    98   0   0   1
299    99   0   0   1

[300 rows x 4 columns]
===============================Model==========================================
                            OLS Regression Results                            
Dep. Variable:                outcome   R-squared:                       0.311
Model:                            OLS   Adj. R-squared:                  0.304
Method:                 Least Squares   F-statistic:                     44.55
Date:                Sun, 31 May 2026   Prob (F-statistic):           8.72e-24
Time:                        21:50:13   Log-Likelihood:                -216.89
No. Observations:                 300   AIC:                             441.8
Df Residuals:                     296

,time,G1,G2,G3,time_m,G3_m
0,0,0.882026,1.441575,0.065409,-49.5,-0.649975
1,1,0.210079,-0.163880,0.140310,-48.5,-0.575073
2,2,0.509369,-0.115242,0.819830,-47.5,0.104446
3,3,1.150447,1.014698,0.607632,-46.5,-0.107752
4,4,0.973779,-0.046562,0.610066,-45.5,-0.105318
...,...,...,...,...,...,...
95,95,1.303287,1.364227,1.768446,45.5,1.053062
96,96,0.965250,1.845895,1.258862,46.5,0.543479
97,97,1.862935,1.881752,1.511477,47.5,0.796093
98,98,1.043456,2.561618,1.030275,48.5,0.314892


# Question 3

In [14]:
hw2 = pd.read_csv('homework_2.2.csv')
hw2

,X,Y,Z
0,0,1.182435,-0.725820
1,0,2.714474,0.563476
2,0,0.077612,-0.435632
3,0,-0.154449,-0.104553
4,0,22.298992,-2.321273
...,...,...,...
9995,0,0.019371,-0.409462
9996,0,2.581533,0.545860
9997,0,0.209599,-0.486216
9998,0,16.829356,-2.045500


In [22]:
treated = hw2[hw2['X'] == 1].copy().reset_index(drop=True)
control = hw2[hw2['X'] == 0].copy().reset_index(drop=True)

effect = treated['Y'].mean() - control['Y'].mean()
effect


np.float64(2.9207172647231907)

# Question 4

In [ ]:
hw2_bs = hw2.sample(n=10000,replace=True, random_state=42)


treated2 = hw2_bs[hw2_bs['X'] == 1].copy().reset_index(drop=True)
control2 = hw2_bs[hw2_bs['X'] == 0].copy().reset_index(drop=True)

effect2 = treated2['Y'].mean() - control2['Y'].mean()
effect2

np.float64(2.7162581901747194)

In [25]:
X_4 = sm.add_constant(hw2.drop('Y',axis=1))
m4 = sm.OLS(hw2['Y'],X_4).fit()
skew_4 = stats.skew(m4.resid)
skew_4


np.float64(2.6879317513980503)

In [1]:
def pareto_bootstrap(bootstrap_size,pareto_sample):
    mean = []
    for sample in bootstrap_size:
        new_sample = np.random.choice(pareto_sample, size=len(pareto_sample),replace=True)
        mean.append(np.mean(new_sample))
    variance = np.var(mean)
    return variance
